# 课后练习解答（04.02_yolo_model_preparation_and_om_conversion）

本解答对应《YOLO 模型准备与 OM 转换》课后练习，共 15 题。


### 问题1（单选题）

**题目：** 本实验中 ATC 转换的直接目标是什么？

A. 把 VOC XML 转成 YOLO txt
B. 把 YOLOv5s ONNX 模型转换为 Ascend 310B4 可执行的 OM 模型
C. 把 bus.jpg 转成权重文件
D. 生成 MindStudio Profiling 报告

**解答：** B

**解析：** ATC 是 Ascend 模型转换工具，本节核心是将 ONNX 转换为 OM。


### 问题2（单选题）

**题目：** YOLOv5s 模型输入在本实验配置中采用的格式是？

A. NHWC
B. NCHW
C. HWC
D. CHWN

**解答：** B

**解析：** 配置中 input_format 为 NCHW，输入 shape 为 [1,3,640,640]。


### 问题3（单选题）

**题目：** 针对 Atlas 200I DK A2 / Ascend 310B4，ATC 的 `--soc_version` 应设置为？

A. Ascend910
B. Ascend310B4
C. Ascend310P3
D. CPU

**解答：** B

**解析：** soc_version 必须与目标芯片匹配，否则 OM 可能无法在开发板上运行。


### 问题4（单选题）

**题目：** `models/yolov5s_310b4.om` 与 `bus.jpg` 的关系最准确的是？

A. OM 是由 bus.jpg 训练得到的
B. OM 是模型文件，bus.jpg 是后续推理验证输入，二者不是同一种文件
C. bus.jpg 会被 ATC 转换成 OM
D. OM 只保存 bus.jpg 的检测框

**解答：** B

**解析：** OM 来自 ONNX 模型转换，bus.jpg 只是用于推理和后处理验证的样例图片。


### 问题5（多选题）

**题目：** ATC 转换时通常需要关注哪些参数？

A. --model
B. --output
C. --input_shape
D. --soc_version

**解答：** A、B、C、D

**解析：** 这些参数分别指定输入模型、输出前缀、输入形状和目标芯片。


### 问题6（多选题）

**题目：** 模型转换前应检查哪些内容？

A. ONNX 文件是否存在
B. CANN 环境变量是否生效
C. 输入 shape 与配置是否一致
D. 目标芯片型号是否正确

**解答：** A、B、C、D

**解析：** 转换失败常见原因就是模型路径、环境、shape 或芯片型号不匹配。


### 问题7（多选题）

**题目：** YOLOv5s OM 输出 shape `[1,25200,85]` 中，85 通常包含哪些信息？

A. 4 个边框坐标
B. 1 个目标置信度
C. 80 个 COCO 类别分数
D. NPU 温度

**解答：** A、B、C

**解析：** YOLOv5s COCO 输出为 4+1+80，和设备状态无关。


### 问题8（判断题）

**题目：** 只要 ONNX 文件存在，ATC 转换一定成功，不需要 CANN 环境。

**解答：** 错误

**解析：** ATC 属于 CANN 工具链，转换需要正确安装并加载 CANN 环境。


### 问题9（判断题）

**题目：** OM 模型是面向 Ascend 设备部署的离线模型格式。

**解答：** 正确

**解析：** OM 是 Ascend 推理部署常用的离线模型产物。


### 问题10（填空题）

**题目：** 本实验配置中的模型输入 shape 是 `____`。

**解答：** [1, 3, 640, 640]

**解析：** 该 shape 对应 batch=1、RGB 三通道、640x640 输入。


### 问题11（填空题）

**题目：** ATC 转换成功后，输出文件通常以 `____` 作为后缀。

**解答：** .om

**解析：** OM 是 Ascend 离线模型文件后缀。


### 问题12（简答题）

**题目：** 为什么 ONNX 到 OM 的转换不能随便换 soc_version？

**解答：** OM 中包含面向目标 Ascend 芯片的图优化和算子编译信息。如果 soc_version 与开发板芯片不匹配，后续 PyACL 加载或执行可能失败。

**解析：** 模型转换和部署设备必须保持一致。


### 问题13（简答题）

**题目：** 为什么本实验使用 bus.jpg 作为样例输入？

**解答：** bus.jpg 是 YOLOv5 常用示例图片，包含多个 COCO 类别目标，适合快速验证预处理、OM 推理、后处理 NMS 和可视化流程。它不参与 ATC 转换，只参与推理验证。

**解析：** 样例图片用于检验模型运行链路，而不是生成模型。


### 问题14（简答题）

**题目：** 转换后如何初步判断 OM 文件是可用于后续实验的？

**解答：** 应检查 OM 文件存在、大小非零、路径与 yolo_edge.yaml 一致，并在后续 PyACL 单元中能够被 acl.mdl.load_from_file 成功加载。

**解析：** 文件存在只是第一步，能被运行时加载才说明链路继续可用。


### 问题15（代码设计题）

**题目：** 写一条 ATC 转换命令，将 `models/yolov5s.onnx` 转为 Ascend310B4 的 OM。

**解答：**

```bash
atc --framework=5 \
  --model=models/yolov5s.onnx \
  --output=models/yolov5s_310b4 \
  --input_shape="images:1,3,640,640" \
  --input_format=NCHW \
  --soc_version=Ascend310B4 \
  --precision_mode=allow_mix_precision
```

**解析：** 实际命令可通过 src/scripts/export_onnx_to_om.sh 封装执行。
